# Kernel agent: Triton k-means assign kernel

Before running anything:
1. **Runtime > Change runtime type > GPU** (A100, L4 or H100; T4 also works but has no bf16).
2. Click the **key icon** in the left sidebar (Secrets), add `ANTHROPIC_API_KEY` with your API key, and turn on
   *Notebook access*. Only step 3 (the agent loop) needs it.

Run the cells top to bottom with **Shift+Enter**. Lines starting with `!` are shell commands.
Files live in `/content` and are deleted when the runtime disconnects, so download `runs.zip` at the end.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, triton; print('torch', torch.__version__, '| triton', triton.__version__)"
!which ncu || ls /usr/local/cuda/bin | grep -iE "ncu|nsys" || echo "ncu not on PATH"

## Write the scripts
Each of the next three cells writes one file to `/content`.

In [ ]:
%%writefile kmeans_bench.py
"""Correctness, benchmark and profiling harness for Double-P's Triton k-means assign kernel.

The kernel is copied from sparse_attention/index/cluster/kmeans.py (_triton_assign_kernel). Shapes mirror how
segment_k_means calls it for LLaMA-3.1-8B (8 kv heads, head_dim 128, ~16 keys per cluster, 1 segment per 8K tokens).

    python kmeans_bench.py check                  # correctness + determinism vs fp32 reference, per dtype
    python kmeans_bench.py bench                  # time Triton vs unfused PyTorch, achieved TFLOP/s per shape
    python kmeans_bench.py iter                   # one full Lloyd iteration: zeroing vs assign (E) vs update (M)
    python kmeans_bench.py jit                    # first-call (compile) latency across new sequence lengths
    python kmeans_bench.py search SHAPE           # config search gated on correctness; reports best vs original
    python kmeans_bench.py ncu VARIANT SHAPE [k=v ...]   # one warmup + one measured launch, for Nsight Compute
"""

import os
import random
import statistics
import sys
import tempfile
import time

# fresh compile cache so first-call timings include real JIT compilation
os.environ.setdefault("TRITON_CACHE_DIR", tempfile.mkdtemp(prefix="triton_cache_"))

import torch  # noqa: E402
import triton  # noqa: E402
import triton.language as tl  # noqa: E402

DEV = "cuda"

# dense (no sparsity) tensor-core peak TFLOP/s per input format and DRAM GB/s; first name match wins.
# "fp32" is TF32 on sm_80+ (Triton's default for fp32 tl.dot) and plain FMA on Turing.
GPU_TABLE = [
    ("H100 PCIe", {"bf16": 756, "fp16": 756, "fp8": 1513, "fp32": 378}, 2000),
    ("H100", {"bf16": 989, "fp16": 989, "fp8": 1979, "fp32": 494}, 3350),
    ("A100", {"bf16": 312, "fp16": 312, "fp32": 156}, 1555),  # 2039 GB/s on the 80 GB part, set below
    ("L4", {"bf16": 121, "fp16": 121, "fp8": 242, "fp32": 60}, 300),
    ("T4", {"fp16": 65, "fp32": 8.1}, 320),
]
DTYPES = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": torch.float32, "fp8": torch.float8_e4m3fn}


def gpu_info():
    p = torch.cuda.get_device_properties(0)
    peaks, dram = {}, None
    for key, pk, bw in GPU_TABLE:
        if key in p.name:
            peaks, dram = pk, bw
            break
    if "A100" in p.name and "80GB" in p.name:
        dram = 2039
    return {
        "name": p.name, "cc": (p.major, p.minor), "sms": p.multi_processor_count,
        "mem_gb": round(p.total_memory / 1e9, 1), "l2_mb": round(getattr(p, "L2_cache_size", 0) / 2**20, 1),
        "peak_tflops": peaks, "dram_gbs": dram,
    }


def supported_formats(cc):
    fmts = ["fp16", "fp32"]
    if cc >= (8, 0):
        fmts.append("bf16")  # bf16 tensor cores arrived with Ampere (A100)
    if cc >= (8, 9):
        fmts.append("fp8")  # fp8 e4m3 tensor cores: Ada (L4) and Hopper (H100)
    return fmts


GPU = gpu_info() if torch.cuda.is_available() else None
# input format: bf16 where the hardware has it (what Double-P stores keys in), otherwise fp16.
# Override with KMEANS_DTYPE=fp16|bf16|fp32|fp8.
DTYPE_NAME = os.environ.get("KMEANS_DTYPE") or ("bf16" if GPU and GPU["cc"] >= (8, 0) else "fp16")
DTYPE = DTYPES[DTYPE_NAME]


def peak_tflops(fmt=None):
    return ((GPU or {}).get("peak_tflops") or {}).get(fmt or DTYPE_NAME)


def describe_gpu():
    g = GPU
    peak = peak_tflops()
    return (f"{g['name']} (sm_{g['cc'][0]}{g['cc'][1]}, {g['sms']} SMs, {g['mem_gb']} GB, L2 {g['l2_mb']} MB, "
            f"DRAM ~{g['dram_gbs'] or '?'} GB/s, {DTYPE_NAME} tensor-core peak ~{peak or '?'} TFLOP/s dense)")

# name: (batch rows, tokens per row, centroids per row) for head_dim 128
SHAPES = {
    "seg-32K": (8 * 4, 8192, 512),  # iterations 1-9 at 32K: 8 kv heads x 4 segments
    "final-32K": (8, 32768, 2048),  # final global pass at 32K
    "seg-128K": (8 * 16, 8192, 512),  # iterations 1-9 at 128K: 16 segments
    "final-128K": (8, 131072, 8192),  # final global pass at 128K
}
HEAD_DIM = 128


# ---------------------------------------------------------------------------------------------------------------------
# original kernel (verbatim logic from kmeans.py): every size and stride is tl.constexpr
# ---------------------------------------------------------------------------------------------------------------------
@triton.jit
def assign_orig(
    K, X, S, C, M,
    stride_kz: tl.constexpr, stride_kn: tl.constexpr, stride_kd: tl.constexpr,
    stride_xz: tl.constexpr, stride_xk: tl.constexpr, stride_xd: tl.constexpr,
    stride_sz: tl.constexpr, stride_sk: tl.constexpr, stride_sd: tl.constexpr,
    stride_cz: tl.constexpr, stride_ck: tl.constexpr,
    stride_mz: tl.constexpr, stride_mn: tl.constexpr,
    num_tokens: tl.constexpr, num_centroids: tl.constexpr,
    BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr, BLOCK_D: tl.constexpr,
):
    start_n = tl.program_id(0) * BLOCK_N
    batch_idx = tl.program_id(1)
    if start_n >= num_tokens:
        return
    offs_n = start_n + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    offs_d = tl.arange(0, BLOCK_D)
    n_mask = offs_n < num_tokens
    k_ptrs = K + batch_idx * stride_kz + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
    x_ptrs = X + batch_idx * stride_xz + offs_k[None, :] * stride_xk + offs_d[:, None] * stride_xd
    s_ptrs = S + batch_idx * stride_sz + offs_d[None, :] * stride_sd
    c_ptrs = C + batch_idx * stride_cz
    m_ptrs = M + batch_idx * stride_mz + offs_n * stride_mn
    k = tl.load(k_ptrs, mask=n_mask[:, None], other=0.0)
    max_val = tl.zeros([BLOCK_N], dtype=tl.float32) - float("inf")
    max_idx = tl.zeros([BLOCK_N], dtype=tl.int32)
    for start_k in tl.range(0, num_centroids, BLOCK_K):
        x = tl.load(x_ptrs)
        ip = tl.dot(k, x).to(tl.float32)
        tmp_max_val, tmp_max_idx = tl.max(ip, axis=1, return_indices=True)
        tmp_max_idx += start_k
        max_idx = tl.where(tmp_max_val > max_val, tmp_max_idx, max_idx)
        max_val = tl.maximum(tmp_max_val, max_val)
        x_ptrs += BLOCK_K * stride_xk
    tl.store(m_ptrs, max_idx, mask=n_mask)
    tl.atomic_add(s_ptrs + max_idx[:, None] * stride_sk, k.to(tl.float32), mask=n_mask[:, None], sem="relaxed")
    tl.atomic_add(c_ptrs + max_idx * stride_ck, tl.zeros_like(max_idx) + 1, mask=n_mask, sem="relaxed")


# ---------------------------------------------------------------------------------------------------------------------
# variant: identical body, but sizes and strides are runtime arguments (no recompile per sequence length)
# ---------------------------------------------------------------------------------------------------------------------
@triton.jit
def assign_dyn(
    K, X, S, C, M,
    stride_kz, stride_kn, stride_kd,
    stride_xz, stride_xk, stride_xd,
    stride_sz, stride_sk, stride_sd,
    stride_cz, stride_ck,
    stride_mz, stride_mn,
    num_tokens, num_centroids,
    BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr, BLOCK_D: tl.constexpr,
):
    start_n = tl.program_id(0) * BLOCK_N
    batch_idx = tl.program_id(1)
    if start_n >= num_tokens:
        return
    offs_n = start_n + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    offs_d = tl.arange(0, BLOCK_D)
    n_mask = offs_n < num_tokens
    k_ptrs = K + batch_idx * stride_kz + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
    x_ptrs = X + batch_idx * stride_xz + offs_k[None, :] * stride_xk + offs_d[:, None] * stride_xd
    s_ptrs = S + batch_idx * stride_sz + offs_d[None, :] * stride_sd
    c_ptrs = C + batch_idx * stride_cz
    m_ptrs = M + batch_idx * stride_mz + offs_n * stride_mn
    k = tl.load(k_ptrs, mask=n_mask[:, None], other=0.0)
    max_val = tl.zeros([BLOCK_N], dtype=tl.float32) - float("inf")
    max_idx = tl.zeros([BLOCK_N], dtype=tl.int32)
    for start_k in tl.range(0, num_centroids, BLOCK_K):
        x = tl.load(x_ptrs)
        ip = tl.dot(k, x).to(tl.float32)
        tmp_max_val, tmp_max_idx = tl.max(ip, axis=1, return_indices=True)
        tmp_max_idx += start_k
        max_idx = tl.where(tmp_max_val > max_val, tmp_max_idx, max_idx)
        max_val = tl.maximum(tmp_max_val, max_val)
        x_ptrs += BLOCK_K * stride_xk
    tl.store(m_ptrs, max_idx, mask=n_mask)
    tl.atomic_add(s_ptrs + max_idx[:, None] * stride_sk, k.to(tl.float32), mask=n_mask[:, None], sem="relaxed")
    tl.atomic_add(c_ptrs + max_idx * stride_ck, tl.zeros_like(max_idx) + 1, mask=n_mask, sem="relaxed")


# ---------------------------------------------------------------------------------------------------------------------
# M-step kernel (verbatim logic from kmeans.py _triton_update_kernel): centroid = sum / count, optional L2 normalize
# ---------------------------------------------------------------------------------------------------------------------
@triton.jit
def update_orig(
    X, S, C,
    stride_xz: tl.constexpr, stride_xk: tl.constexpr, stride_xd: tl.constexpr,
    stride_sz: tl.constexpr, stride_sk: tl.constexpr, stride_sd: tl.constexpr,
    stride_cz: tl.constexpr, stride_ck: tl.constexpr,
    num_centroids: tl.constexpr,
    BLOCK_K: tl.constexpr, BLOCK_D: tl.constexpr, NORMORLIZE: tl.constexpr,
):
    start_k = tl.program_id(0) * BLOCK_K
    batch_idx = tl.program_id(1)
    offs_k = start_k + tl.arange(0, BLOCK_K)
    offs_d = tl.arange(0, BLOCK_D)
    k_mask = offs_k < num_centroids
    x_ptrs = X + batch_idx * stride_xz + offs_k[:, None] * stride_xk + offs_d[None, :] * stride_xd
    s_ptrs = S + batch_idx * stride_sz + offs_k[:, None] * stride_sk + offs_d[None, :] * stride_sd
    c_ptrs = C + batch_idx * stride_cz + offs_k[:, None] * stride_ck
    s = tl.load(s_ptrs, mask=k_mask[:, None], other=0.0)
    c = tl.load(c_ptrs, mask=k_mask[:, None], other=0)
    x_mask = c > 0
    x = s / c
    if NORMORLIZE:
        x /= tl.sqrt(tl.sum(x * x, axis=-1, keep_dims=True))
    tl.store(x_ptrs, x.to(X.type.element_ty), mask=x_mask)


def launch_update(cent, s, cnt, normalize=True, block_k=128):
    b, c, d = cent.shape
    update_orig[(triton.cdiv(c, block_k), b, 1)](
        cent, s, cnt, *cent.stride(), *s.stride(), *cnt.stride(), c,
        BLOCK_K=block_k, BLOCK_D=d, NORMORLIZE=normalize, num_warps=4, num_stages=1,
    )


KERNELS = {"orig": assign_orig, "dyn": assign_dyn}
DEFAULT_CFG = {"block_n": 128, "block_k": 32, "num_warps": 4, "num_stages": 2}  # what kmeans.py uses


def make_inputs(b, n, c, dtype=None, seed=0):
    dtype = dtype or DTYPE
    g = torch.Generator(device=DEV).manual_seed(seed)
    data = torch.randn(b, n, HEAD_DIM, device=DEV, generator=g).to(dtype)
    cent = torch.randn(b, c, HEAD_DIM, device=DEV, generator=g)
    cent = (cent / cent.norm(dim=-1, keepdim=True)).to(dtype)  # spherical k-means keeps unit centroids
    return data, cent


def alloc_outputs(data, cent):
    b, n, d = data.shape
    c = cent.shape[1]
    s = torch.zeros(b, c, d, device=DEV, dtype=torch.float32)
    cnt = torch.zeros(b, c, device=DEV, dtype=torch.int32)
    m = torch.empty(b, n, device=DEV, dtype=torch.int32)
    return s, cnt, m


def launch(variant, data, cent, s, cnt, m, block_n=128, block_k=32, num_warps=4, num_stages=2):
    b, n, d = data.shape
    c = cent.shape[1]
    assert c % block_k == 0, "centroid loads are unmasked: num_centroids must be a multiple of BLOCK_K"
    KERNELS[variant][(triton.cdiv(n, block_n), b, 1)](
        data, cent, s, cnt, m,
        *data.stride(), *cent.stride(), *s.stride(), *cnt.stride(), *m.stride(),
        n, c,
        BLOCK_N=block_n, BLOCK_K=block_k, BLOCK_D=d, num_warps=num_warps, num_stages=num_stages,
    )


def run(variant, data, cent, **cfg):
    s, cnt, m = alloc_outputs(data, cent)
    launch(variant, data, cent, s, cnt, m, **cfg)
    return s, cnt, m


def torch_assign(data, cent):
    # unfused reference: GEMM -> argmax -> scatter-add, materializing the [B, N, C] score matrix
    ip = torch.bmm(data, cent.transpose(1, 2))
    m = ip.argmax(-1)
    b, c, d = cent.shape
    s = torch.zeros(b, c, d, device=DEV, dtype=torch.float32).scatter_add_(
        1, m[..., None].expand(-1, -1, d), data.float())
    cnt = torch.zeros(b, c, device=DEV, dtype=torch.float32).scatter_add_(1, m, torch.ones_like(m, dtype=torch.float32))
    return s, cnt.int(), m


def verify(data, cent, s, cnt, m, max_tokens=4096):
    """Compare a Triton result against exact fp32 math. Returns a dict of metrics."""
    n_chk = min(data.shape[1], max_tokens)
    d32, c32 = data[:, :n_chk].float(), cent.float()
    ip32 = torch.bmm(d32, c32.transpose(1, 2))
    ref = ip32.argmax(-1)
    mm = m[:, :n_chk].long()
    best = ip32.gather(-1, ref[..., None]).squeeze(-1)
    got = ip32.gather(-1, mm[..., None]).squeeze(-1)
    knorm = d32.norm(dim=-1).clamp_min(1e-30)
    # sums / counts must be consistent with the kernel's own assignment
    b, c, d = cent.shape
    s_ref = torch.zeros(b, c, d, device=DEV, dtype=torch.float64).scatter_add_(
        1, m.long()[..., None].expand(-1, -1, d), data.float().double())  # via fp32: fp8 has no direct cast
    cnt_ref = torch.zeros(b, c, device=DEV, dtype=torch.float64).scatter_add_(
        1, m.long(), torch.ones_like(m, dtype=torch.float64))
    return {
        "argmax_agree": (mm == ref).double().mean().item(),
        # how much worse the chosen centroid's score is than the true best (0 = exact or tie)
        "max_regret": (best - got).max().item(),
        # the same gap divided by |key|: a cosine-similarity gap, since centroids are unit norm
        "max_rel_regret": ((best - got) / knorm).max().item(),
        "sum_rel_err": ((s.double() - s_ref).abs().max() / s_ref.abs().max()).item(),
        "counts_exact": bool(torch.equal(cnt.double(), cnt_ref)),
    }


# chosen centroid's score within REL_TOL * |key| of the best: 16x finer than bf16's 2^-8 relative precision. Scale-free,
# unlike an absolute tolerance (real keys range from |k| ~20 to ~800), and loose enough for near-tie flips and for
# approximations whose error is provably below it.
REL_TOL = 2.0 ** -12


def passes(v, rel_tol=REL_TOL):
    return v["counts_exact"] and v["sum_rel_err"] < 1e-5 and v["max_rel_regret"] <= rel_tol


def flops_and_bytes(b, n, c, elem):
    flops = 2.0 * b * n * c * HEAD_DIM
    min_bytes = b * n * HEAD_DIM * elem + b * c * HEAD_DIM * elem + b * n * 4  # keys + centroids once + labels
    atomic_bytes = b * n * HEAD_DIM * 4  # one fp32 atomic add per key element
    return flops, min_bytes, atomic_bytes


def bench_fn(fn):
    return triton.testing.do_bench(fn, warmup=25, rep=100)


def interleaved(fns, zero, rounds=7):
    """Time each fn in rotating order for `rounds` rounds, after holding the GPU busy so clocks settle.

    Each fn zeroes the accumulators and then launches, as a real k-means iteration does (this also leaves s in L2 the
    way the kernel finds it in practice). `zero` alone is timed in the same rounds and subtracted, so the returned
    {name: [ms per round]} is kernel time only. Also returns the per-round zeroing times.
    """
    fns = {**fns, "_zero": zero}
    names = list(fns)
    for f in fns.values():  # compile first
        f()
    triton.testing.do_bench(fns[names[0]], warmup=500, rep=500)
    raw = {k: [] for k in names}
    for r in range(rounds):
        for k in names[r % len(names):] + names[:r % len(names)]:
            raw[k].append(bench_fn(fns[k]))
    zeros = raw.pop("_zero")
    return {k: [t - z for t, z in zip(ts, zeros)] for k, ts in raw.items()}, zeros


def timed_launch(variant, data, cent, s, cnt, m, **cfg):
    """Zero the accumulators, then launch; the timed unit for interleaved()."""
    def f():
        s.zero_()
        cnt.zero_()
        launch(variant, data, cent, s, cnt, m, **cfg)
    return f


def zeroer(s, cnt):
    def f():
        s.zero_()
        cnt.zero_()
    return f


def bench_kernel(variant, data, cent, s, cnt, m, rounds=3, **cfg):
    """Median kernel-only ms (zeroing subtracted) and median zeroing ms."""
    t, zeros = interleaved({"k": timed_launch(variant, data, cent, s, cnt, m, **cfg)}, zeroer(s, cnt), rounds)
    return statistics.median(t["k"]), statistics.median(zeros)


# ---------------------------------------------------------------------------------------------------------------------
def cmd_check():
    b, n, c = 8, 8192, 512
    print(f"GPU: {describe_gpu()}")
    print(f"correctness at B={b}, N={n}, C={c}, D={HEAD_DIM} vs exact fp32 math (first 4096 tokens per row)")
    fmts = supported_formats(GPU["cc"])
    if GPU["cc"] < (8, 0):
        print("  bf16 skipped: no bf16 tensor cores before sm_80; fp32 runs on CUDA cores (no TF32 before sm_80)")
    for fmt in fmts:
        label = fmt + (" (TF32 dot)" if fmt == "fp32" and GPU["cc"] >= (8, 0) else "")
        data, cent = make_inputs(b, n, c, DTYPES[fmt])
        try:
            s, cnt, m = run("orig", data, cent, **DEFAULT_CFG)
            torch.cuda.synchronize()
        except Exception as e:
            print(f"  {label:16s} FAILED to compile/run: {type(e).__name__}: {str(e).splitlines()[0][:100]}")
            continue
        v = verify(data, cent, s, cnt, m)
        s2, cnt2, m2 = run("orig", data, cent, **DEFAULT_CFG)
        ms, _ = bench_kernel("orig", data, cent, *alloc_outputs(data, cent), **DEFAULT_CFG)
        tflops = 2.0 * b * n * c * HEAD_DIM / ms / 1e9
        print(f"  {label:16s} agree={v['argmax_agree']:.5f} regret={v['max_regret']:.2e} "
              f"sum_err={v['sum_rel_err']:.1e} counts_exact={v['counts_exact']}  "
              f"labels_identical_rerun={torch.equal(m, m2)} "
              f"sum_bits_differ_rerun={(s != s2).double().mean().item():.2%}  "
              f"time={ms:.3f} ms ({tflops:.1f} TFLOP/s, peak {peak_tflops(fmt) or '?'})")


def cmd_bench():
    print(f"GPU: {describe_gpu()}   input format: {DTYPE_NAME}")
    print(f"{'shape':11s} {'B':>4s} {'N':>7s} {'C':>5s} | {'triton ms':>9s} {'TFLOP/s':>7s} {'%peak':>6s} "
          f"{'zero us':>7s} | {'torch ms':>9s} {'speedup':>7s}")
    elem = torch.empty(0, dtype=DTYPE).element_size()
    peak = peak_tflops()
    for name, (b, n, c) in SHAPES.items():
        data, cent = make_inputs(b, n, c)
        ms, zero_ms = bench_kernel("orig", data, cent, *alloc_outputs(data, cent), **DEFAULT_CFG)
        flops, _, _ = flops_and_bytes(b, n, c, elem)
        tflops = flops / ms / 1e9
        pct = f"{tflops / peak:6.1%}" if peak else f"{'?':>6s}"
        # unfused PyTorch materializes the [B, N, C] score matrix; skip when it would not fit comfortably
        score_bytes = b * n * c * 2
        if DTYPE_NAME != "fp8" and score_bytes < 0.25 * GPU["mem_gb"] * 1e9:
            tms = bench_fn(lambda: torch_assign(data, cent))
            tcell = f"{tms:9.3f} {tms / (ms + zero_ms):6.2f}x"  # torch_assign allocates its own zeroed outputs
        else:
            tcell = f"{'skip':>9s} {'-':>7s}"
        print(f"{name:11s} {b:4d} {n:7d} {c:5d} | {ms:9.3f} {tflops:7.2f} {pct} {zero_ms * 1e3:7.1f} | {tcell}")


def cmd_iter():
    # one Lloyd iteration exactly as _triton_k_means_train runs it: allocate zeroed accumulators, assign, update
    print(f"GPU: {describe_gpu()}   input format: {DTYPE_NAME}")
    print(f"{'shape':11s} | {'zeros us':>8s} {'assign ms':>9s} {'update us':>9s} | {'iteration ms':>12s} "
          f"{'assign share':>12s} {'other':>8s}")
    for name, (b, n, c) in SHAPES.items():
        data, cent = make_inputs(b, n, c)
        work = cent.clone()  # the update writes centroids in place; keep the inputs untouched
        s, cnt, m = alloc_outputs(data, cent)
        d = data.shape[-1]

        def zeros():
            torch.zeros(b, c, d, device=DEV, dtype=torch.float32)
            torch.zeros(b, c, device=DEV, dtype=torch.int32)

        def full_iteration():
            s_ = torch.zeros(b, c, d, device=DEV, dtype=torch.float32)
            cnt_ = torch.zeros(b, c, device=DEV, dtype=torch.int32)
            launch("orig", data, work, s_, cnt_, m, **DEFAULT_CFG)
            launch_update(work, s_, cnt_)

        launch("orig", data, cent, s, cnt, m, **DEFAULT_CFG)  # realistic counts for the update timing
        t_zero = bench_fn(zeros)
        t_assign = bench_fn(lambda: launch("orig", data, cent, s, cnt, m, **DEFAULT_CFG))
        t_update = bench_fn(lambda: launch_update(work, s, cnt))
        t_iter = bench_fn(full_iteration)
        other = t_iter - t_assign
        print(f"{name:11s} | {t_zero * 1e3:8.1f} {t_assign:9.3f} {t_update * 1e3:9.1f} | {t_iter:12.3f} "
              f"{t_assign / t_iter:12.1%} {other * 1e3:6.1f}us")


def cmd_jit():
    random.seed(0)
    lengths = [random.randint(6000, 12000) for _ in range(5)]
    c = 512
    for variant in ["orig", "dyn"]:
        firsts, steadies = [], []
        for n in lengths:
            data, cent = make_inputs(8, n, c)
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            run(variant, data, cent, **DEFAULT_CFG)
            torch.cuda.synchronize()
            firsts.append((time.perf_counter() - t0) * 1e3)
            t0 = time.perf_counter()
            run(variant, data, cent, **DEFAULT_CFG)
            torch.cuda.synchronize()
            steadies.append((time.perf_counter() - t0) * 1e3)
        print(f"{variant:5s} new lengths {lengths}")
        print(f"      first call ms: {[round(x, 1) for x in firsts]}")
        print(f"      steady call ms: {[round(x, 2) for x in steadies]}")


def cmd_search(shape):
    b, n, c = SHAPES[shape]
    data, cent = make_inputs(b, n, c)
    s, cnt, m = alloc_outputs(data, cent)
    rows = []
    for block_n in [64, 128, 256]:
        for block_k in [32, 64, 128]:
            if c % block_k:
                continue
            for num_warps in [4, 8]:
                for num_stages in [1, 2, 3, 4]:
                    cfg = dict(block_n=block_n, block_k=block_k, num_warps=num_warps, num_stages=num_stages)
                    try:
                        vs, vcnt, vm = run("dyn", data, cent, **cfg)
                        torch.cuda.synchronize()
                    except Exception as e:  # out of shared memory / registers
                        rows.append((cfg, None, f"launch failed: {type(e).__name__}"))
                        continue
                    v = verify(data, cent, vs, vcnt, vm)
                    if not passes(v):
                        rows.append((cfg, None, f"REJECTED: {v}"))
                        continue
                    ms = bench_fn(timed_launch("dyn", data, cent, s, cnt, m, **cfg))  # coarse: zeroing included
                    rows.append((cfg, ms, f"agree={v['argmax_agree']:.4f}"))
    ok = sorted([r for r in rows if r[1] is not None], key=lambda r: r[1])
    bad = [r for r in rows if r[1] is None]
    print(f"{shape}: {len(ok)} configs passed the correctness gate, {len(bad)} failed or were rejected")
    for cfg, _, note in bad[:5]:
        print(f"  x {cfg}: {note[:120]}")

    # the coarse pass times configs minutes apart while clocks drift, so re-time the top 5 against the original in
    # interleaved rounds; dyn at the default config and orig at the best config separate the variant from the config
    top = ok[:5]
    fns = {"orig default": timed_launch("orig", data, cent, s, cnt, m, **DEFAULT_CFG),
           "dyn default": timed_launch("dyn", data, cent, s, cnt, m, **DEFAULT_CFG),
           "orig best cfg": timed_launch("orig", data, cent, s, cnt, m, **top[0][0])}
    for cfg, _, _ in top:
        fns[f"dyn {cfg}"] = timed_launch("dyn", data, cent, s, cnt, m, **cfg)
    t, zeros = interleaved(fns, zeroer(s, cnt))
    print(f"re-timed in interleaved rounds (kernel time, zeroing {statistics.median(zeros) * 1e3:.1f} us subtracted), "
          f"vs the original {DEFAULT_CFG}:")
    for name, ts in t.items():
        ratios = sorted(o / x for o, x in zip(t["orig default"], ts))
        print(f"  {statistics.median(ts):8.3f} ms  {statistics.median(ratios):6.3f}x "
              f"(rounds {ratios[0]:.3f}-{ratios[-1]:.3f})  {name}")


def cmd_ncu(variant, shape, overrides):
    b, n, c = SHAPES[shape]
    cfg = dict(DEFAULT_CFG)
    for kv in overrides:
        k, v = kv.split("=")
        cfg[k] = int(v)
    data, cent = make_inputs(b, n, c)
    run(variant, data, cent, **cfg)  # launch 1: compile + warmup (skip it in ncu)
    torch.cuda.synchronize()
    run(variant, data, cent, **cfg)  # launch 2: the one to profile
    torch.cuda.synchronize()
    print(f"profiled {variant} on {shape} with {cfg}")


if __name__ == "__main__":
    mode = sys.argv[1] if len(sys.argv) > 1 else "check"
    if mode == "check":
        cmd_check()
    elif mode == "bench":
        cmd_bench()
    elif mode == "iter":
        cmd_iter()
    elif mode == "jit":
        cmd_jit()
    elif mode == "search":
        cmd_search(sys.argv[2] if len(sys.argv) > 2 else "final-32K")
    elif mode == "ncu":
        cmd_ncu(sys.argv[2], sys.argv[3], sys.argv[4:])
    else:
        sys.exit(__doc__)

In [ ]:
%%writefile eval_candidate.py
"""Evaluate one candidate implementation of the k-means assign kernel and print one JSON line.

A candidate is a Python module defining `launch(data, cent, s, cnt, m)` that fills the outputs using Triton kernels.
Runs in its own process so a crashing or hanging kernel cannot take down the agent loop.

    python eval_candidate.py CANDIDATE.py SHAPE                  # correctness gate + benchmark vs the original kernel
    python eval_candidate.py CANDIDATE.py SHAPE --real GATE.pt   # also gate on real keys at SHAPE, and time on them
    python eval_candidate.py CANDIDATE.py SHAPE --profile        # one warmup + one launch in an NVTX range, for ncu
    python eval_candidate.py CANDIDATE.py --heldout HELDOUT.pt   # correctness on held-out real keys only, no timing
"""

import argparse
import importlib.util
import json
import re
import statistics
import sys
import traceback
from pathlib import Path

import torch

sys.path.insert(0, str(Path(__file__).resolve().parent))
import kmeans_bench as kb  # noqa: E402

# (batch rows, tokens, centroids): ragged token counts exercise the masking paths
CHECK_SHAPES = [(3, 1000, 128), (2, 4173, 512)]
ROUNDS = 7  # interleaved candidate/original timing rounds
# torch-side compute that would bypass Triton (tl.argmax and friends are allowed)
FORBIDDEN = [r"torch\.matmul", r"torch\.bmm", r"torch\.einsum", r"torch\.mm\(", r"(?<!tl)\.argmax\(", r"scatter_add",
             r"torch_assign", r"kmeans_bench", r"F\.linear", r"(?<![\w.])@(?![\w.])", r"tensordot", r"addb?mm",
             r"torch\.inner", r"torch\.dot", r"torch\.mv\("]


def load(path):
    spec = importlib.util.spec_from_file_location("candidate", path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.launch


def load_real(path):
    """Real keys file from make_real_keys.py: keys [B, N, D], unit-norm centroids [B, C, D], one (layer, head) per row."""
    d = torch.load(path, map_location="cpu")
    return d["keys"].to(kb.DEV, kb.DTYPE), d["cent"].to(kb.DEV, kb.DTYPE), [tuple(r) for r in d["rows"]]


def outputs_ok(data, cent, s, cnt, m, c, max_tokens=4096):
    b, n, _ = data.shape
    if int(m.min()) < 0 or int(m.max()) >= c:
        return {"shape": [b, n, c], "error": f"labels out of range [{int(m.min())}, {int(m.max())}]"}, False
    v = kb.verify(data, cent, s, cnt, m, max_tokens=max_tokens)
    return {"shape": [b, n, c], **v}, kb.passes(v)


def check(launch, b, n, c, seed):
    data, cent = kb.make_inputs(b, n, c, seed=seed)
    s, cnt, m = kb.alloc_outputs(data, cent)
    launch(data, cent, s, cnt, m)
    torch.cuda.synchronize()
    return outputs_ok(data, cent, s, cnt, m, c)


def rows_ok(keys, cents, s, cnt, m, rows):
    """Per (layer, head) row, every token checked."""
    results, ok = [], True
    for i, (layer, head) in enumerate(rows):
        sl = slice(i, i + 1)
        res, passed = outputs_ok(keys[sl], cents[sl], s[sl], cnt[sl], m[sl], cents.shape[1], max_tokens=1 << 30)
        res.pop("shape")
        results.append({"layer": layer, "head": head, **res})
        ok &= passed
    return results, ok


def real_rows(launch, keys, cents, rows):
    s, cnt, m = kb.alloc_outputs(keys, cents)
    launch(keys, cents, s, cnt, m)
    torch.cuda.synchronize()
    return rows_ok(keys, cents, s, cnt, m, rows)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("candidate")
    ap.add_argument("shape", nargs="?")
    ap.add_argument("--real", help="real keys at SHAPE: added to the gate and used for timing and profiling")
    ap.add_argument("--heldout", help="only check correctness on these real keys")
    ap.add_argument("--profile", action="store_true")
    args = ap.parse_args()
    out = {"candidate": args.candidate, "shape": args.shape, "passed": False}
    try:
        src = Path(args.candidate).read_text()
        if not Path(args.candidate).name.endswith("_baseline.py"):  # only harness-written baselines may call kmeans_bench
            bad = [f for f in FORBIDDEN if re.search(f, src)]
            if bad:
                out["error"] = f"forbidden constructs (compute must be in Triton kernels): {bad}"
                print(json.dumps(out))
                return
        launch = load(args.candidate)

        if args.heldout:
            res, ok = real_rows(launch, *load_real(args.heldout))
            out.update(passed=ok, heldout=res)
            print(json.dumps(out))
            return

        b, n, c = kb.SHAPES[args.shape]
        rows = None
        if args.real:
            data, cent, rows = load_real(args.real)
            assert (data.shape[0], data.shape[1], cent.shape[1]) == (b, n, c), "real keys must be at the target shape"
        else:
            data, cent = kb.make_inputs(b, n, c)
        s, cnt, m = kb.alloc_outputs(data, cent)

        if args.profile:
            launch(data, cent, s, cnt, m)  # compile + warmup
            torch.cuda.synchronize()
            s.zero_()
            cnt.zero_()
            torch.cuda.nvtx.range_push("measure")
            launch(data, cent, s, cnt, m)
            torch.cuda.synchronize()
            torch.cuda.nvtx.range_pop()
            return

        checks, ok = [], True
        for i, (cb, cn, cc) in enumerate(CHECK_SHAPES + [(b, n, c)]):
            res, passed = check(launch, cb, cn, cc, seed=100 + i)
            checks.append(res)
            ok &= passed
        out["checks"] = checks
        if rows:
            out["real_checks"], real_ok = real_rows(launch, data, cent, rows)
            ok &= real_ok
        if not ok:
            out["error"] = "correctness gate failed (see checks" + (" and real_checks)" if rows else ")")
            print(json.dumps(out))
            return

        def cand():
            s.zero_()
            cnt.zero_()
            launch(data, cent, s, cnt, m)

        base = kb.timed_launch("orig", data, cent, s, cnt, m, **kb.DEFAULT_CFG)

        # GPU clocks drift between processes by more than most candidate differences, so time the candidate and the
        # original in rotating order in this process and score the median per-round ratio of kernel times
        t, zeros = kb.interleaved({"cand": cand, "base": base}, kb.zeroer(s, cnt), rounds=ROUNDS)

        # the timed calls are never checked, so check one more call that looks exactly like them: a candidate that
        # skips work on repeated calls would otherwise pass the gate and time near zero
        m.fill_(-1)
        cand()
        torch.cuda.synchronize()
        res, ok = rows_ok(data, cent, s, cnt, m, rows) if rows else outputs_ok(data, cent, s, cnt, m, c)
        if not ok:
            out.update(error="outputs wrong on a repeated call after timing (every call must do the full work)",
                       repeat_check=res)
            print(json.dumps(out))
            return

        ratios = sorted(bt / ct for bt, ct in zip(t["base"], t["cand"]))
        ms, base_ms = statistics.median(t["cand"]), statistics.median(t["base"])
        zero_ms = statistics.median(zeros)
        flops, _, _ = kb.flops_and_bytes(b, n, c, data.element_size())
        peak = kb.peak_tflops()
        tflops = flops / ms / 1e9
        out.update(passed=True, ms=ms, baseline_ms=base_ms, zero_ms=zero_ms, speedup=statistics.median(ratios),
                   speedup_range=[ratios[0], ratios[-1]],
                   speedup_incl_zeroing=(base_ms + zero_ms) / (ms + zero_ms), tflops=tflops,
                   pct_of_peak=(tflops / peak) if peak else None, dtype=kb.DTYPE_NAME,
                   timed_on="real keys" if rows else "synthetic")
    except Exception as e:  # compile errors, launch errors, illegal memory access
        out["error"] = f"{type(e).__name__}: {e}"[:2000]
        out["traceback"] = traceback.format_exc()[-3000:]
    print(json.dumps(out))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile agent_loop.py
"""Agent loop for kernel optimization: Claude proposes Triton kernel variants, a harness gates each one on correctness
against an fp32 reference, benchmarks it against the original kernel, optionally profiles it with Nsight Compute, and
feeds the results back. The fastest correct kernel is kept. No human in the loop between iterations.

Colab (A100 / L4 / H100 GPU runtime; T4 also works with fp16), with kmeans_bench.py and eval_candidate.py in the
same folder:
    !pip install -q -U anthropic
    import os; from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")   # add the key under Colab "Secrets"
    !python agent_loop.py --shape final-32K --iters 8 --ncu

With real keys from make_real_keys.py (gate on and time on one set, report on a held-out set after the run):
    !python agent_loop.py --shape final-32K --iters 15 --patience 6 --ncu \
        --real-gate real_keys_gate.pt --heldout real_keys_heldout.pt [--seed previous_run/iter06.py]
"""

import argparse
import json
import re
import subprocess
import sys
import time
from pathlib import Path

import anthropic

HERE = Path(__file__).resolve().parent
MODEL = "claude-opus-5-5"
# USD per million tokens for claude-opus-5-5 (input, output, cache read, cache write ~1.25x input)
PRICE = {"in": 4.0, "out": 20.0, "cache_read": 0.20, "cache_write": 5.0}

BASELINE_SRC = '''# BASELINE: the original kernel from sparse_attention/index/cluster/kmeans.py, via kmeans_bench.py
import kmeans_bench as kb


def launch(data, cent, s, cnt, m):
    kb.launch("orig", data, cent, s, cnt, m, **kb.DEFAULT_CFG)
'''

TUNED_SRC = '''# TUNED BASELINE: the best launch config from the brute-force search, on the kernel variant the search used
import kmeans_bench as kb


def launch(data, cent, s, cnt, m):
    kb.launch("dyn", data, cent, s, cnt, m, **{cfg})
'''

SYSTEM_TEMPLATE = """You are a GPU kernel engineer optimizing one Triton kernel. Each turn you receive measurements \
from an automated harness and return one new candidate implementation. There is no human in the loop.

## Hardware
{hardware}

## Operation (k-means assignment step with fused accumulation)
Inputs: data [B, N, D] {dtype} contiguous (keys), cent [B, C, D] {dtype} contiguous (unit-norm centroids), D = 128.
Outputs (zeroed / allocated by the harness before every call):
- m [B, N] int32: m[b, n] = argmax_c dot(data[b, n], cent[b, c])
- s [B, C, D] fp32 (zeroed): s[b, c] += data[b, n] for every n assigned to c
- cnt [B, C] int32 (zeroed): cnt[b, c] += 1 for every n assigned to c
Target shape: {shape} = (B, N, C) = {bnc}. C is always a multiple of 128; N may be any positive integer.

## Contract
Return a complete, self-contained Python module that imports only torch, triton and triton.language and defines
    def launch(data, cent, s, cnt, m): ...
which fills s, cnt and m using Triton kernels. All compute must happen in Triton kernels (no torch matmul, \
argmax or scatter ops; allocating scratch buffers with torch is fine). You may change anything else: tiling, \
num_warps, num_stages, loop structure, number of kernels, persistent scheduling, how atomics are issued, etc.

## Correctness gate (a candidate that fails is discarded)
Checked against exact fp32 math on synthetic inputs, (3, 1000, 128), (2, 4173, 512) and the target shape (randn keys, \
unit-norm randn centroids){real_gate}:
- labels in range, and for every checked key the chosen centroid's score is within {rel_tol:.3g} * |key| of the best \
score (a cosine-similarity gap, since centroids are unit norm; 16x finer than bf16's relative precision). Near-tie \
flips pass; approximate scoring passes only if its error on these keys stays below that bound.
- s matches the sum of data rows for the kernel's own labels (relative error < 1e-5)
- cnt matches exactly
- after timing, one more call on the timing inputs is checked the same way, so every call must do the full work

## Objective
Maximize speedup on the target shape{timed_on}: the median over interleaved rounds, in one process, of (original \
kernel time / candidate kernel time). Each timed call zeroes s and cnt and then launches (triton.testing.do_bench); \
the zeroing alone is timed in the same rounds and subtracted, so ms, speedup and tflops are kernel-only. Absolute \
times drift between processes with GPU clocks, so compare candidates by speedup, and treat differences inside the \
reported speedup_range as noise.

## Reference implementation and harness
The harness (kmeans_bench.py) is below. `assign_orig` with DEFAULT_CFG is the baseline you must beat.

```python
{harness}
```

## Response format
1. At most 6 sentences: the bottleneck you infer from the latest measurements and the single main change you \
are making to address it.
2. Exactly one ```python fenced block containing the complete candidate module.
"""


REAL_GATE_NOTE = """, and on real post-RoPE keys from Qwen2.5-1.5B at exactly the target shape: 8 rows, each one \
(layer, head) pair with 32768 keys and 2048 unit-length centroids, every key checked. The keys are prepared the way \
Double-P clusters them (first 4 and last 64 prompt tokens dropped, the head's mean key subtracted) and the centroids \
come from Double-P's segment k-means. Real keys have larger outlier channels than Gaussian ones, and neighbouring \
tokens often share a centroid. Per-row results come back as real_checks. A held-out set of real \
keys from other layers, heads and prompts is checked after the run and never shown to you, so do not special-case \
the gate data"""


def evaluate(path, shape, real=None, timeout=900):
    cmd = [sys.executable, str(HERE / "eval_candidate.py"), str(path), shape] + (["--real", real] if real else [])
    return run_json(cmd, timeout)


def heldout(path, data_path, timeout=900):
    return run_json([sys.executable, str(HERE / "eval_candidate.py"), str(path), "--heldout", data_path], timeout)


def run_json(cmd, timeout):
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return {"passed": False, "error": f"timed out after {timeout}s (hang or very slow compile)"}
    lines = [ln for ln in proc.stdout.splitlines() if ln.startswith("{")]
    if not lines:
        return {"passed": False, "error": f"evaluator crashed (exit {proc.returncode}): {proc.stderr[-2000:]}"}
    return json.loads(lines[-1])


def profile(path, shape, real=None, timeout=900):
    cmd = ["ncu", "--nvtx", "--nvtx-include", "measure/", "--section", "SpeedOfLight", "--section", "Occupancy",
           "--section", "LaunchStats", "--section", "WarpStateStats",
           sys.executable, str(HERE / "eval_candidate.py"), str(path), shape, "--profile"]
    cmd += ["--real", real] if real else []
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except (FileNotFoundError, subprocess.TimeoutExpired) as e:
        return f"(profiler unavailable: {type(e).__name__})"
    text = proc.stdout
    start = text.find("==PROF==")
    text = text[start:] if start >= 0 else text
    if "ERR_NVGPUCTRPERM" in text or "No kernels were profiled" in text:
        return "(profiler could not collect counters on this machine)"
    return text[-7000:]


def extract_code(text):
    blocks = re.findall(r"```python\n(.*?)```", text, flags=re.S)
    return blocks[-1] if blocks else None


def summarize(result):
    keep = {k: result[k] for k in ["passed", "ms", "baseline_ms", "zero_ms", "speedup", "speedup_range",
                                   "speedup_incl_zeroing", "tflops", "pct_of_peak", "dtype", "timed_on", "error",
                                   "checks", "real_checks", "repeat_check"]
            if k in result}
    if "traceback" in result and not result.get("passed"):
        keep["traceback_tail"] = result["traceback"][-1500:]
    return json.dumps(keep, indent=1, default=str)


def call_claude(client, system, messages, effort):
    with client.beta.messages.stream(
        model=MODEL,
        max_tokens=32000,
        system=system,
        messages=messages,
        thinking={"type": "adaptive", "display": "summarized"},
        output_config={"effort": effort},
        cache_control={"type": "ephemeral"},  # caches the growing conversation prefix turn over turn
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",  # re-run on a fallback model if a safety classifier declines
    ) as stream:
        return stream.get_final_message()


def cost(usage):
    return (usage.input_tokens * PRICE["in"] + usage.output_tokens * PRICE["out"]
            + (usage.cache_read_input_tokens or 0) * PRICE["cache_read"]
            + (usage.cache_creation_input_tokens or 0) * PRICE["cache_write"]) / 1e6


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--shape", default="final-32K")
    ap.add_argument("--iters", type=int, default=8)
    ap.add_argument("--patience", type=int, default=4, help="stop after this many iterations without improvement")
    ap.add_argument("--effort", default="high", choices=["low", "medium", "high", "xhigh", "max"])
    ap.add_argument("--ncu", action="store_true", help="feed Nsight Compute sections back to the model")
    ap.add_argument("--tuned", nargs="*", default=[], metavar="KEY=VAL",
                    help="best launch config a config search found for the original kernel (e.g. num_stages=4); "
                         "the agent is told about it and has to beat it")
    ap.add_argument("--real-gate", help="real keys at the target shape (make_real_keys.py): gated on and timed on")
    ap.add_argument("--heldout", help="held-out real keys, checked for every passing candidate after the run only")
    ap.add_argument("--seed", help="a kernel from an earlier run to start from; its code and results go to the agent")
    ap.add_argument("--hint", default="", help="extra guidance added to the agent's first message")
    ap.add_argument("--out", default=str(HERE / "runs"), help="parent folder for run folders (e.g. on Google Drive)")
    args = ap.parse_args()

    sys.path.insert(0, str(HERE))
    import kmeans_bench as kb

    run_dir = Path(args.out) / time.strftime("%Y%m%d-%H%M%S")
    run_dir.mkdir(parents=True)
    log = open(run_dir / "log.jsonl", "w")
    system = SYSTEM_TEMPLATE.format(shape=args.shape, bnc=kb.SHAPES[args.shape], hardware=kb.describe_gpu(),
                                    dtype=kb.DTYPE_NAME, harness=(HERE / "kmeans_bench.py").read_text(),
                                    rel_tol=kb.REL_TOL, real_gate=REAL_GATE_NOTE if args.real_gate else "",
                                    timed_on=", timed on the real gate keys" if args.real_gate else "")
    print(f"GPU: {kb.describe_gpu()}   input format: {kb.DTYPE_NAME}", flush=True)

    base_path = run_dir / "iter00_baseline.py"
    base_path.write_text(BASELINE_SRC)
    base = evaluate(base_path, args.shape, args.real_gate)
    if not base.get("passed"):
        sys.exit(f"baseline failed: {summarize(base)}")
    base_prof = profile(base_path, args.shape, args.real_gate) if args.ncu else "(profiling disabled)"
    lo, hi = base["speedup_range"]
    print(f"baseline: {base['ms']:.3f} ms ({base['tflops']:.2f} TFLOP/s); noise floor, original vs itself: "
          f"{base['speedup']:.3f}x (rounds {lo:.3f}-{hi:.3f})", flush=True)
    best = {"iter": 0, "speedup": 1.0, "path": base_path, "name": "the baseline"}
    passing = [(best["name"], base_path, 1.0)]  # every gate-passing kernel, for the held-out check at the end

    tuned, tuned_note = None, ""
    if args.tuned:
        cfg = {**kb.DEFAULT_CFG, **{k: int(v) for k, v in (kv.split("=") for kv in args.tuned)}}
        tuned_path = run_dir / "iter00_tuned_baseline.py"  # the _baseline.py suffix exempts it from the import check
        tuned_path.write_text(TUNED_SRC.format(cfg=cfg))
        tuned = evaluate(tuned_path, args.shape, args.real_gate)
        if not tuned.get("passed"):
            sys.exit(f"tuned baseline failed: {summarize(tuned)}")
        print(f"tuned baseline {cfg}: {tuned['ms']:.3f} ms, {tuned['speedup']:.3f}x vs original", flush=True)
        best = {"iter": 0, "speedup": tuned["speedup"], "path": tuned_path, "name": "the tuned baseline"}
        passing.append((best["name"], tuned_path, tuned["speedup"]))
        tuned_note = (f"A brute-force search over launch configs found {cfg} on the runtime-size variant (assign_dyn), "
                      f"measured at {tuned['speedup']:.3f}x vs the original. Config tuning alone reaches that, so a "
                      "candidate only counts if its speedup is higher.\n\n")

    seed_note = ""
    if args.seed:
        seed_src = Path(args.seed).read_text()
        seed_path = run_dir / "iter00_seed.py"
        seed_path.write_text(seed_src)
        seed = evaluate(seed_path, args.shape, args.real_gate)
        if seed.get("passed"):
            print(f"seed kernel {args.seed}: {seed['ms']:.3f} ms, {seed['speedup']:.3f}x vs original", flush=True)
            passing.append(("the seed kernel", seed_path, seed["speedup"]))
            if seed["speedup"] > best["speedup"]:
                best = {"iter": 0, "speedup": seed["speedup"], "path": seed_path, "name": "the seed kernel"}
        else:
            print(f"seed kernel {args.seed} FAILED the gate: {seed.get('error', '')[:120]}", flush=True)
        seed_note = ("Starting point: a kernel from an earlier run, measured under this gate:\n"
                     f"{summarize(seed)}\n\n```python\n{seed_src}```\n\n")

    client = anthropic.Anthropic()
    messages = [{"role": "user", "content": (
        f"Baseline measurements on {args.shape}:\n{summarize(base)}\n\nNsight Compute (baseline):\n{base_prof}\n\n"
        f"{tuned_note}{seed_note}{args.hint + chr(10) * 2 if args.hint else ''}Propose candidate 1.")}]
    total_cost, stale = 0.0, 0
    for it in range(1, args.iters + 1):
        try:
            resp = call_claude(client, system, messages, args.effort)
        except anthropic.RateLimitError as e:
            print(f"rate limited, stopping: {e.message}")
            break
        except anthropic.APIStatusError as e:
            print(f"API error {e.status_code}, stopping: {e.message}")
            break
        except anthropic.APIConnectionError:
            print("network error, stopping")
            break
        messages.append({"role": "assistant", "content": resp.content})  # full content, append-only history
        total_cost += cost(resp.usage)
        if resp.stop_reason == "refusal":
            print(f"iteration {it}: request declined, stopping")
            break

        text = "".join(b.text for b in resp.content if b.type == "text")
        reasoning = "".join(b.thinking for b in resp.content if b.type == "thinking")
        code = extract_code(text)
        cand_path = run_dir / f"iter{it:02d}.py"
        if code is None:
            result = {"passed": False, "error": "no ```python block found in the response"}
        else:
            cand_path.write_text(code)
            result = evaluate(cand_path, args.shape, args.real_gate)
        prof = profile(cand_path, args.shape, args.real_gate) if (args.ncu and result.get("passed")) else ""

        improved = result.get("passed") and result["speedup"] > best["speedup"]
        if improved:
            best = {"iter": it, "speedup": result["speedup"], "path": cand_path, "name": f"candidate {it}"}
            stale = 0
        else:
            stale += 1
        if result.get("passed"):
            passing.append((f"candidate {it}", cand_path, result["speedup"]))
            lo, hi = result["speedup_range"]
            status = f"{result['ms']:.3f} ms, {result['speedup']:.3f}x vs original (rounds {lo:.3f}-{hi:.3f})"
            if tuned:
                status += f", {result['speedup'] / tuned['speedup']:.3f}x vs tuned"
        else:
            status = f"FAILED: {result.get('error', '')[:120]}"
        print(f"iter {it:2d}: {status}{'  <- best' if improved else ''}   (API cost so far ${total_cost:.2f})",
              flush=True)
        log.write(json.dumps({"iter": it, "result": result, "hypothesis": text.split("```")[0].strip(),
                              "reasoning_summary": reasoning, "usage": resp.usage.to_dict()}, default=str) + "\n")
        log.flush()
        if stale >= args.patience:
            print(f"no improvement for {args.patience} iterations, stopping")
            break

        messages.append({"role": "user", "content": (
            f"Candidate {it} results:\n{summarize(result)}\n\n"
            + (f"Nsight Compute (candidate {it}):\n{prof}\n\n" if prof else "")
            + f"Best so far: {best['name']} at {best['speedup']:.3f}x vs the original"
            + (f" (tuned baseline {tuned['speedup']:.3f}x)" if tuned else "") + ".\n"
            f"Propose candidate {it + 1}.")})

    (run_dir / "best_kernel.py").write_text(Path(best["path"]).read_text())
    print(f"\nbest: {best['name']} at {best['speedup']:.3f}x vs the original"
          + (f", {best['speedup'] / tuned['speedup']:.3f}x vs the tuned baseline" if tuned else "")
          + f". Files in {run_dir}. API cost ~${total_cost:.2f}")

    if args.heldout:
        # the test set: never shown to the agent, so it shows whether passing the gate generalizes
        print(f"\nheld-out real keys ({args.heldout}), every gate-passing kernel:")
        with open(run_dir / "heldout.jsonl", "w") as f:
            for name, path, speedup in passing:
                r = heldout(path, args.heldout)
                rows = r.get("heldout", [])
                worst = max((row.get("max_rel_regret", float("inf")) for row in rows), default=float("inf"))
                failed = [f"L{row['layer']}h{row['head']}" for row in rows
                          if not (row.get("counts_exact") and row.get("max_rel_regret", 1) <= kb.REL_TOL
                                  and row.get("sum_rel_err", 1) < 1e-5)]
                verdict = "pass" if r.get("passed") else f"FAIL {' '.join(failed) or r.get('error', '')[:80]}"
                mark = "  <- best" if Path(path) == Path(best["path"]) else ""
                print(f"  {name:18s} {speedup:6.3f}x  worst cos gap {worst:.1e}  {verdict}{mark}", flush=True)
                f.write(json.dumps({"name": name, "path": str(path), "speedup": speedup, **r}, default=str) + "\n")


if __name__ == "__main__":
    main()

## Step 1: correctness and number formats
Compares input formats against exact fp32 math (agreement, regret, sum error) and times each one.
`fp32` runs as TF32 inside `tl.dot` on A100/L4/H100. `fp8` appears only on L4/H100.

In [ ]:
!python kmeans_bench.py check

In [ ]:
!python kmeans_bench.py bench

Where one full k-means iteration spends its time: zeroing the accumulators, assign (E-step), update (M-step).

In [ ]:
!python kmeans_bench.py iter

## Step 2: profile the original kernel with Nsight Compute
If this prints `ERR_NVGPUCTRPERM` or `ncu: not found`, counters are unavailable on this runtime;
skip this step and drop `--ncu` in step 3.

In [ ]:
!ncu --kernel-name regex:assign --launch-skip 1 --launch-count 1 --section SpeedOfLight --section ComputeWorkloadAnalysis --section MemoryWorkloadAnalysis --section Occupancy --section WarpStateStats --section LaunchStats python kmeans_bench.py ncu orig final-32K

Optional: brute-force config search (no LLM), gated on the same correctness check.

In [ ]:
!python kmeans_bench.py search final-32K

## Step 3: agent loop (phase 2: real keys)
First upload `real_keys_gate.pt` and `real_keys_heldout.pt` (from the `kernel-agent` folder on your Desktop) to
`MyDrive/kernel-agent/` in Google Drive.

- The correctness gate also checks real Qwen2.5-1.5B keys at the timed shape, and candidates are timed on them.
- Correctness is relative: the chosen centroid must score within 2^-12 x |key| of the best.
- After the run, every passing kernel is checked on held-out keys (other layers, heads and prompts) that the agent never sees.
- `SEED` starts the agent from the previous run's last full-precision kernel; set it to `""` to start from scratch.

Runs are saved to Drive, so a Colab disconnect does not delete them. Roughly $3-5 for 15 iterations.
Drop `--ncu` if step 2 failed.

In [ ]:
!pip install -q -U anthropic
import os
from google.colab import drive, userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/kernel-agent"
OUT = f"{DRIVE}/runs"
GATE, HELDOUT = f"{DRIVE}/real_keys_gate.pt", f"{DRIVE}/real_keys_heldout.pt"
SEED = f"{OUT}/20261026-004515/iter06.py"  # previous run's last full-precision kernel; "" to start from scratch
os.makedirs(OUT, exist_ok=True)
for p in [GATE, HELDOUT] + ([SEED] if SEED else []):
    print("ok     " if os.path.exists(p) else "MISSING", p)

In [ ]:
cmd = f"python agent_loop.py --shape final-32K --iters 15 --patience 6 --ncu --real-gate {GATE} --heldout {HELDOUT} --out {OUT}"
if SEED:
    cmd += f" --seed {SEED}"
!{cmd}

## Step 4: inspect the best kernel (works after a disconnect too: rerun the Drive cell above first)

In [ ]:
import glob, json
runs = sorted(glob.glob(f"{OUT}/*"))
assert runs, f"no runs in {OUT}"
run = runs[-1]
print(run)
print(open(f"{run}/best_kernel.py").read())
for line in open(f"{run}/log.jsonl"):
    r = json.loads(line)
    res = r["result"]
    status = f"{res['ms']:.3f} ms ({res['speedup']:.2f}x)" if res.get("passed") else "failed: " + res.get("error", "")[:80]
    print(f"iter {r['iter']:2d}: {status}\n    {r['hypothesis'][:300]}")

Same-process comparison on synthetic, gate and held-out keys. GPU clocks drift between processes by up to ~20%, so
every kernel is timed here in interleaved rounds (order rotated each round) and scored by its median per-round ratio
to the original. Times are kernel-only: each call zeroes the accumulators first, and the zeroing is timed separately
and subtracted. The input matters: on real keys neighbouring tokens often get the same centroid, so the atomic
`s[label] += key` adds collide, while synthetic Gaussian keys almost never share a label within a tile.

In [ ]:
import importlib.util, statistics, kmeans_bench as kb, eval_candidate as ec
PREV = f"{OUT}/20261026-004515"  # phase-1 run: iter06 (full precision) and the int8 best

def load_launch(path):
    spec = importlib.util.spec_from_file_location(path, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.launch

kernels = {"phase 1 iter06 (the seed)": load_launch(f"{PREV}/iter06.py"),
           "phase 1 best (int8, fails real keys)": load_launch(f"{PREV}/best_kernel.py"),
           "phase 2 best": load_launch(f"{run}/best_kernel.py")}

def compare(data, cent, kernels, rounds=9):
    """{name: (median kernel ms, median per-round ratio vs the original, sorted ratios)}"""
    s, cnt, m = kb.alloc_outputs(data, cent)
    fns = {"original": kb.timed_launch("orig", data, cent, s, cnt, m, **kb.DEFAULT_CFG)}
    for name, launch in kernels.items():
        def f(launch=launch):
            s.zero_()
            cnt.zero_()
            launch(data, cent, s, cnt, m)
        fns[name] = f
    t, _ = kb.interleaved(fns, kb.zeroer(s, cnt), rounds=rounds)
    out = {}
    for k, ts in t.items():
        r = sorted(o / x for o, x in zip(t["original"], ts))
        out[k] = (statistics.median(ts), statistics.median(r), r)
    return out

b, n, c = kb.SHAPES["final-32K"]
for label, (data, cent) in [("synthetic", kb.make_inputs(b, n, c)), ("gate keys", ec.load_real(GATE)[:2]),
                            ("held-out keys", ec.load_real(HELDOUT)[:2])]:
    print(label)
    for k, (ms, x, r) in compare(data, cent, kernels).items():
        print(f"  {k:38s} {ms:.3f} ms  {x:.3f}x vs original (rounds {r[0]:.3f}-{r[-1]:.3f})")

One (layer, head) at a time: the row is copied into all 8 batch slots, so the shape, and each kernel's chosen config,
stay the same. Layer 0 is the degenerate one, where nearly every key lands in one cluster.

In [ ]:
pick = {k: kernels[k] for k in ["phase 1 iter06 (the seed)", "phase 2 best"]}
for path in [GATE, HELDOUT]:
    keys, cents, rows = ec.load_real(path)
    for i, (layer, head) in enumerate(rows):
        res = compare(keys[i:i + 1].expand(b, -1, -1).contiguous(), cents[i:i + 1].expand(b, -1, -1).contiguous(),
                      pick, rounds=5)
        print(f"L{layer:2d}h{head}  original {res['original'][0]:.3f} ms | "
              + " | ".join(f"{k} {v[0]:.3f} ms {v[1]:.2f}x" for k, v in res.items() if k != "original"))

## Step 5: per-row detail on the held-out keys
The loop already printed a pass/fail table for every passing kernel; this shows each (layer, head) row for the
original and the best kernel.

In [ ]:
import eval_candidate as ec
keys, cents, rows = ec.load_real(HELDOUT)
for name, fn in [("orig", lambda *a: kb.launch("orig", *a, **kb.DEFAULT_CFG)), ("agent best", kernels["phase 2 best"])]:
    res, ok = ec.real_rows(fn, keys, cents, rows)
    print(name, "PASS" if ok else "FAIL")
    for r in res:
        print(f"  layer {r['layer']:2d} head {r['head']}: agree={r.get('argmax_agree', float('nan')):.5f} "
              f"cos gap={r.get('max_rel_regret', float('nan')):.1e} {r.get('error', '')}")

Correctness of the best kernel against exact fp32 math (agreement, worst regret, sum error) on the gate shapes, 5 seeds each.

In [ ]:
import eval_candidate as ec
for cb, cn, cc in ec.CHECK_SHAPES + [kb.SHAPES["final-32K"]]:
    for seed in range(5):
        res, ok = ec.check(kernels["phase 2 best"], cb, cn, cc, seed=200 + seed)
        print("PASS" if ok else "FAIL", res)

## Step 6: third pass, from iter06, on centered keys
The first gate and held-out files used raw keys, but Double-P drops the first 4 and last 64 prompt tokens and
subtracts each head's mean key before clustering. On raw keys, layer 0 piled ~99% of keys into one cluster, a case the
kernel never sees in Double-P. `test.ipynb` (on your Mac) writes centered versions to `centered/`. Upload both files to
`MyDrive/kernel-agent/centered/`. That keeps the old raw files, which `kernel_explained.ipynb` still reads.

Run the Drive cell in step 3 and the three `%%writefile` cells first. The first cell below checks the files and runs
iter06 through the same harness the loop uses: gate correctness and timing vs the original, then held-out correctness.
Expect about 1.36x. The second cell starts the loop from iter06. Afterwards, steps 4 and 5 show this run, since they
pick the latest run folder.

The loop cell also gives the agent `HINT`: phase 2's 6-bit sub-tile packing, which cuts iter06's error limit from
2^-12 (exactly the tolerance) to 2^-17. Phase 2 spent iteration 9 finding it. Set `HINT = ""` to leave it out.

In [ ]:
import json, os, subprocess, sys, torch
KEYS_DIR = f"{DRIVE}/centered"  # set to DRIVE if you replaced the old files instead
GATE, HELDOUT = f"{KEYS_DIR}/real_keys_gate.pt", f"{KEYS_DIR}/real_keys_heldout.pt"
SEED = f"{OUT}/20261026-004515/iter06.py"
for p in [GATE, HELDOUT]:
    meta = torch.load(p, map_location="cpu")["model"]
    assert "mean-centered" in meta, f"{p} holds the old raw keys ({meta})"
    print("ok", p, "|", meta)
assert os.path.exists(SEED), f"missing {SEED}"

def harness(*args):
    p = subprocess.run([sys.executable, "eval_candidate.py", *args], capture_output=True, text=True)
    if not p.stdout.strip():
        raise RuntimeError(p.stderr[-3000:])
    return json.loads(p.stdout.strip().splitlines()[-1])

r = harness(SEED, "final-32K", "--real", GATE)
print(f"gate: {'PASS' if r['passed'] else 'FAIL ' + r.get('error', '')[:200]}")
if r["passed"]:
    print(f"  iter06 {r['ms']:.3f} ms vs original {r['baseline_ms']:.3f} ms: {r['speedup']:.3f}x "
          f"(rounds {r['speedup_range'][0]:.3f}-{r['speedup_range'][1]:.3f}), timed on {r['timed_on']}")
for row in r.get("real_checks", []):
    print(f"  L{row['layer']:2d}h{row['head']} cos gap {row.get('max_rel_regret', float('nan')):.1e}")
h = harness(SEED, "--heldout", HELDOUT)
print(f"held-out: {'PASS' if h['passed'] else 'FAIL'}")
for row in h.get("heldout", []):
    print(f"  L{row['layer']:2d}h{row['head']} cos gap {row.get('max_rel_regret', float('nan')):.1e}")

In [ ]:
import shlex
HINT = ("The seed packs the full 11-bit centroid index into the low bits of each fp32 score, leaving 12 fraction "
        "bits, so its worst-case cosine gap is just under 2^-12: exactly the tolerance, with no margin. An earlier "
        "run removed that at no measurable speed cost: keep a [BLOCK_N, BLOCK_K] running best in which column j "
        "always holds centroid sub_tile * BLOCK_K + j, pack only the sub-tile number (6 bits for 64 sub-tiles of "
        "32), and decode at the end with an argmax over the BLOCK_K columns, which leaves 17 fraction bits (limit "
        "2^-17). Build on that, or on something with similar margin.")
cmd = (f"python agent_loop.py --shape final-32K --iters 15 --patience 6 --ncu --real-gate {GATE} "
       f"--heldout {HELDOUT} --out {OUT} --seed {SEED}" + (f" --hint {shlex.quote(HINT)}" if HINT else ""))
!{cmd}